In [74]:
import ModifiedNEAT.nn as mn
import torch
import torch.nn as nn
import torch.nn.functional as F
from ModifiedNEAT import NeatParameter, NeatModule
from torch import Tensor

Embedding layer

In [75]:
class Embedding(NeatModule):
    def __init__(
        self, vocab_size: int, dim_size: int,  padding_idx: int | None = None, 
        device: torch.device = 'cpu', dtype: torch.dtype = torch.float32,
    ):
        super(Embedding, self).__init__(
            vocab_size=vocab_size, dim_size=dim_size, padding_idx=padding_idx
        )

        self.vocab_size = vocab_size
        self.dim_size = dim_size
        self.padding_idx = padding_idx

        # Learnable embedding matrix
        self.weights = NeatParameter((vocab_size, dim_size), False, device, dtype)
        # shape (genomes, vocab_size, dim_size)

        # Optional padding handling
        if padding_idx is not None:
            with torch.no_grad():
                self.weights.data[padding_idx].fill_(0)

    def forward(self, tensor : Tensor | int, keys: int | list[int] = None):
        """
        input_ids: LongTensor of shape (genomes, ...)
        returns: embeddings of shape (genomes, ..., dim_size)
        """
        genomes = torch.tensor(self.weights.get_list(keys), 
            device=self.weights.device, dtype=torch.long)

        if isinstance(tensor, (int, float)):
            tensor = torch.full((genomes.shape[0],), int(tensor),
                device=self.weights.device, dtype=torch.long)
        elif tensor.ndim == 0:
            tensor = tensor.unsqueeze(0).to(self.weights.device)

        # Expand genome indices to match tensor shape
        while genomes.ndim < tensor.ndim:
            genomes = genomes.unsqueeze(-1)

        genomes = genomes.expand_as(tensor)

        # Advanced indexing
        return self.weights.data[genomes, tensor]

In [76]:
genomes = 1
vocab_size = 10
embed_size = 4

In [77]:
test_embedder = Embedding(vocab_size, embed_size, padding_idx=None)
test_embedder, test_embedder.weights.data.shape

(Embedding[NeatModule](vocab_size=10, dim_size=4, padding_idx=None),
 torch.Size([1, 10, 4]))

In [78]:
test_tensor = torch.randint(0, vocab_size, (genomes, 2, 2,),).long()
test_tensor.shape

torch.Size([1, 2, 2])

In [83]:
test_out = test_embedder(7, keys=None)
test_out.shape, test_out

(torch.Size([1, 4]), tensor([[-0.5627, -1.8616,  0.6273,  2.0530]]))